# Biohub - Cell Tracking S1.5 Pipeline

このノートブックは、**S1.5フェーズ (スコアアップ戦略)** を実行するための統合検証ノートブックです。

## プロジェクト構成

```text
. (プロジェクトフォルダ)
├── working/                                   # 作業ディレクトリ
│   ├── s1_05_stardist_btrack_pipeline.ipynb   # main処理ノートブック
│   └── kaggle_cell_tracking_competition/      # 【準備1】主催者の公式リポジトリ
│       ├── README.md
│       ├── pyproject.toml
│       ├── tests/
│       ├── scripts/
│       └── src/
│           └── tracking_cellmot/              # 公式の評価用ライブラリ
└── input/                                     # 【準備2】inputデータ
    ├── test/                                  # 提出用データセット (.zarr / .geff)
    │   ├── xxxx.zarr/
    │   └── xxxx.geff/
    └── train/                                 # 訓練用データセット (.zarr / .geff)
        ├── xxxx.zarr/
        └── xxxx.geff/
```

### 【準備1】主催者の公式リポジトリの準備方法
working/配下でclone実行
```batch
git clone https://github.com/royerlab/kaggle_cell_tracking_competition.git
```

### 【準備2】inputデータの準備方法
`./input/` フォルダ配下に展開します。

** データの入手先:**
* [Biohub - Cell Tracking During Development Data](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/data)で、`Download All` ボタンを押し、ZIPファイルをダウンロード → 解答して展開。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# === オフライン環境でのライブラリ自動インストール ===
import sys
import subprocess

def is_installed(package_name):
    try:
        __import__(package_name)
        return True
    except ImportError:
        return False

def run_pip(cmd_args):
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.system(f"pip {cmd_args}")
            return
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip"] + cmd_args.split(), check=True)

# 1. Zarr パッケージのインストール
if not is_installed("zarr"):
    print("Installing zarr...")
    run_pip("install --no-index --find-links=../input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr")
else:
    print("zarr is already installed.")

# 2. tracksdata 関連パッケージのインストール
if not is_installed("tracksdata") or not is_installed("geff") or not is_installed("polars"):
    print("Installing tracksdata and dependencies...")
    run_pip("install --no-index --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels rustworkx bidict ilpy imagecodecs polars")
    run_pip("install --no-index --no-deps --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels geff geff-spec")
    run_pip("install --no-index --no-deps --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels tracksdata")
else:
    print("tracksdata and dependencies are already installed.")

print("Offline installation steps completed.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Offline installation check completed. (Elapsed: {_cell_elapsed:.2f}s)")


In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

import os
import sys
import glob
import numpy as np
import pandas as pd
from skimage.feature import blob_dog

# === Polarsのエラー回避モンキーパッチ ===
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32

# === パス設定 (静的解決) ===
target_path = os.path.abspath(os.path.join(os.getcwd(), '../input/datasets/aaaa1597/kaggle-cell-tracking-competition', 'src'))
if os.path.exists(os.path.join(target_path, 'tracking_cellmot')):
    if target_path not in sys.path:
        sys.path.insert(0, target_path)
    print(f"Path set successfully: {target_path}")
else:
    fallback_path = os.path.abspath(os.path.join(os.getcwd(), 'src'))
    if fallback_path not in sys.path:
        sys.path.insert(0, fallback_path)
    print(f"Warning: Expected path not found. Using fallback path: {fallback_path}")

import tracksdata as td
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import node_recall, evaluate, evaluate_datasets
from tracksdata.metrics import DistanceMatching
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === GPU有無の自動判定とCPU用モンキーパッチの適用 ===
import torch
if not torch.cuda.is_available():
    print("GPU is not available. Applying CPU monkey patch to tracking_cellmot.io._process_on_gpu...")
    import tracking_cellmot.io

    def patched_process_on_gpu(
        image, tracks, scale, device,
        resample=False, target_scale=None,
        normalize=True, gamma=1.0,
        q_min=0.01, q_max=0.99, subsample_factor=1000,
        precomputed_quantiles=None
    ):
        torch_device = torch.device(device)
        image = image.astype(np.float32)

        q1, q2 = None, None
        if normalize:
            q1 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_min)
            q2 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_max)
            if q1 is None or q2 is None:
                flat = image.ravel()[::subsample_factor]
                q1, q2 = np.quantile(flat, [q_min, q_max]).astype(np.float32)
            else:
                q1 = np.float32(q1)
                q2 = np.float32(q2)

        tensor = torch.from_numpy(image)
        tensor = tensor.to(torch_device, non_blocking=True)

        if normalize:
            tensor = (tensor - float(q1)) / (float(q2) - float(q1) + 1e-6)
            tensor = tensor.clamp(min=0.0)
            if gamma != 1.0:
                tensor = tensor.pow(gamma)
            tensor = tensor.clamp(0.0, 4.0)

        if resample:
            scale_arr = np.array(scale)
            if target_scale is None:
                target_scale_val = scale_arr.min()
            else:
                target_scale_val = np.array(target_scale)

            zoom_factors = scale_arr / target_scale_val
            T = tensor.shape[0]
            new_spatial_shape = (np.array(tensor.shape[1:]) * zoom_factors).astype(int).tolist()
            tensor = tensor[:, None]
            tensor = torch.nn.functional.interpolate(
                tensor, size=new_spatial_shape, mode="trilinear", align_corners=False
            )
            tensor = tensor[:, 0]
            
            if tracks is not None:
                node_attrs = tracks.node_attrs()
                orig_dtypes = {col: node_attrs.schema[col] for col in ["z", "y", "x"]}
                node_attrs = node_attrs.with_columns(
                    (pl.col("z") * zoom_factors[0]).round(0).cast(orig_dtypes["z"]),
                    (pl.col("y") * zoom_factors[1]).round(0).cast(orig_dtypes["y"]),
                    (pl.col("x") * zoom_factors[2]).round(0).cast(orig_dtypes["x"]),
                )
                tracks.update_node_attrs(
                    attrs=node_attrs.select("z", "y", "x").to_dict(),
                    node_ids=node_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),
                )
            if target_scale is not None:
                scale = target_scale
            else:
                scale = (float(target_scale_val),) * 3

        return tensor, tracks, scale

    tracking_cellmot.io._process_on_gpu = patched_process_on_gpu
    print("CPU Monkey Patch applied successfully!")
else:
    print("GPU is available. No monkey patch needed.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Imports and path resolution completed. (Elapsed: {_cell_elapsed:.2f}s)")



## 【ステップ 1 & 2】検出評価と検出器の向上

In [ ]:
def detect_nodes_baseline(dataset_path, min_sigma=2.0, max_sigma=5.0, threshold=0.05, max_frames=None):
    """
    ベースライン検出器 (blob_dog)
    """
    ds = open_dataset(dataset_path, normalize=True, require_tracks=False, device='cpu')
    images = ds.image
    n_frames = images.shape[0]
    from tqdm import tqdm
    if max_frames is not None:
        n_frames = min(n_frames, max_frames)
    
    nodes = []
    global_node_id = 0
    
    for t in tqdm(range(n_frames), desc="Detecting cells in 3D frames"):
        frame = images[t]
        if hasattr(frame, 'numpy'):
            frame = frame.numpy()
        
        img_min, img_max = frame.min(), frame.max()
        if img_max > img_min:
            img_norm = (frame.astype(np.float32) - img_min) / (img_max - img_min)
        else:
            img_norm = np.zeros_like(frame, dtype=np.float32)
            
        blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        
        for blob in blobs:
            z, y, x, r = blob
            nodes.append({
                'node_id': global_node_id,
                't': t,
                'z': float(z),
                'y': float(y),
                'x': float(x)
            })
            global_node_id += 1
            
    return pd.DataFrame(nodes)


def evaluate_detection_only(pred_nodes_df, gt_graph, scale, max_distance=7.0):
    """
    ステップ1: 検出単体の評価 (Recall, Precision, F1-Scoreの内訳)
    """
    pred_graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node({
            't': int(row.t),
            'z': float(row.z),
            'y': float(row.y),
            'x': float(row.x)
        })
        
    matching = DistanceMatching(max_distance=max_distance, scale=scale)
    pred_graph.match(gt_graph, matching=matching)
    
    node_attrs = pred_graph.node_attrs(
        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]
    )
    matched = node_attrs.filter(
        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()
        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)
    )
    
    tp = matched[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].n_unique()
    pred_total = pred_graph.num_nodes()
    fp = pred_total - len(matched)
    
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['t'])
    gt_total_in_range = len(gt_nodes_pl.filter(pl.col('t') < 3)) if 't' in gt_nodes_pl.columns else gt_graph.num_nodes()
    fn = gt_total_in_range - tp
    
    precision = tp / pred_total if pred_total > 0 else 0.0
    recall_local = tp / gt_total_in_range if gt_total_in_range > 0 else 1.0
    recall_global = tp / gt_graph.num_nodes() if gt_graph.num_nodes() > 0 else 1.0
    
    f1_local = 2 * precision * recall_local / (precision + recall_local) if (precision + recall_local) > 0 else 0.0
    f1_global = 2 * precision * recall_global / (precision + recall_global) if (precision + recall_global) > 0 else 0.0
    
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'pred_total': pred_total,
        'gt_total_in_range': gt_total_in_range,
        'precision': precision,
        'recall_local': recall_local,
        'recall_global': recall_global,
        'f1_local': f1_local,
        'f1_global': f1_global
    }

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell Cell detection and evaluation functions defined. (Elapsed: {_cell_elapsed:.2f}s)")


### 1.1 検出結果の3D視覚化 (Error Analysis)

予測された細胞位置 (ノード) と GT を3D空間にプロットし、公式マッチング基準 (7 µm) で正しく検出できたもの (TP: 緑)、余分な検出 (FP: 赤)、見落とした正解 (FN: 青) を色分けして可視化します。

In [ ]:
def plot_detection_dashboard(image_4d, pred_nodes_df, gt_graph, max_distance=7.0, scale=[1.0, 1.0, 1.0]):
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from scipy.spatial import KDTree
    import ipywidgets as widgets
    from IPython.display import display

    scale = np.array(scale)
    n_frames, nz, ny, nx = image_4d.shape
    gt_nodes_df = gt_graph.node_attrs(attr_keys=['t', 'z', 'y', 'x']).to_pandas()

    # 各フレームのデータを事前に計算してキャッシュに格納
    frame_data_cache = []
    
    for t in range(n_frames):
        p_nodes = pred_nodes_df[pred_nodes_df['t'] == t].copy()
        g_nodes = gt_nodes_df[gt_nodes_df['t'] == t].copy()
        
        p_coords = p_nodes[['z', 'y', 'x']].values * scale
        g_coords = g_nodes[['z', 'y', 'x']].values * scale
        
        tp_coords = np.empty((0, 3))
        fp_coords = p_coords if len(p_coords) > 0 else np.empty((0, 3))
        fn_coords = g_coords if len(g_coords) > 0 else np.empty((0, 3))
        
        tp, fp, fn = 0, len(p_nodes), len(g_nodes)
        
        if len(p_coords) > 0 and len(g_coords) > 0:
            tree = KDTree(g_coords)
            distances, indices = tree.query(p_coords, distance_upper_bound=max_distance)
            
            matched_g = set()
            tp_p_idx = []
            tp_g_idx = []
            for p_idx, (d, g_idx) in enumerate(zip(distances, indices)):
                if d <= max_distance and g_idx not in matched_g:
                    tp_p_idx.append(p_idx)
                    tp_g_idx.append(g_idx)
                    matched_g.add(g_idx)
            
            if tp_p_idx:
                tp_coords = p_coords[tp_p_idx]
                fp_coords = np.delete(p_coords, tp_p_idx, axis=0)
                fn_coords = np.delete(g_coords, tp_g_idx, axis=0)
                tp = len(tp_p_idx)
                fp = len(p_nodes) - tp
                fn = len(g_nodes) - tp
                
        precision = tp / len(p_nodes) if len(p_nodes) > 0 else 0.0
        recall = tp / len(g_nodes) if len(g_nodes) > 0 else 1.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # 評価用の装飾テキストHTML
        eval_text = (
            f"<div style='font-family: monospace; background-color: #f8f9fa; border: 1px solid #ccc; padding: 10px; border-radius: 5px; font-size: 13px; line-height: 1.5; color: black;'>"
            f"<b>DETAILED DETECTION EVALUATION (FRAME {t})</b><br>"
            f"  TP (Matched predicted nodes): <span style='color: green; font-weight: bold;'>{tp}</span> | "
            f"  FP (Extra predicted nodes):   <span style='color: red; font-weight: bold;'>{fp}</span> | "
            f"  FN (Missed GT nodes):         <span style='color: blue; font-weight: bold;'>{fn}</span> (GT total in frame: {len(g_nodes)})<br>"
            f"  Precision:          <span style='font-weight: bold;'>{precision:.4f}</span> | "
            f"  Recall (Local):     <span style='font-weight: bold;'>{recall:.4f}</span> | "
            f"  F1-Score:           <span style='font-weight: bold;'>{f1:.4f}</span>"
            f"</div>"
        )
        
        frame_data_cache.append({
            't': t,
            'image_3d': image_4d[t],
            'tp_coords': tp_coords,
            'fp_coords': fp_coords,
            'fn_coords': fn_coords,
            'eval_text': eval_text
        })

    # 特定のフレームの Figure を新しく生成するヘルパー関数
    def create_figure_for_frame(t):
        cache = frame_data_cache[t]
        
        fig = go.Figure(make_subplots(
            rows=2, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}], [{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=(
                "1. 3D Point Cloud",
                "2. XY-MIP 3D Overlay (Z=0.0 Bottom Wall)",
                "3. XZ-MIP 3D Overlay (Y=0.0 Back Wall)",
                "4. YZ-MIP 3D Overlay (X=0.0 Side Wall)"
            )
        ))
        
        mip_z = np.max(cache['image_3d'], axis=0)
        mip_y = np.max(cache['image_3d'], axis=1)
        mip_x = np.max(cache['image_3d'], axis=2)
        
        x_grid_z, y_grid_z = np.meshgrid(np.arange(nx) * scale[2], np.arange(ny) * scale[1])
        x_grid_y, z_grid_y = np.meshgrid(np.arange(nx) * scale[2], np.arange(nz) * scale[0])
        y_grid_x, z_grid_x = np.meshgrid(np.arange(ny) * scale[1], np.arange(nz) * scale[0])
        
        # Subplot 1 (1,1): 3D Point Cloud
        fig.add_trace(go.Scatter3d(x=cache['tp_coords'][:, 2] if len(cache['tp_coords']) > 0 else [], y=cache['tp_coords'][:, 1] if len(cache['tp_coords']) > 0 else [], z=cache['tp_coords'][:, 0] if len(cache['tp_coords']) > 0 else [], mode='markers', name='TP (Correct)', marker=dict(size=4, color='lime', opacity=0.8)), row=1, col=1)
        fig.add_trace(go.Scatter3d(x=cache['fp_coords'][:, 2] if len(cache['fp_coords']) > 0 else [], y=cache['fp_coords'][:, 1] if len(cache['fp_coords']) > 0 else [], z=cache['fp_coords'][:, 0] if len(cache['fp_coords']) > 0 else [], mode='markers', name='FP (Extra)', marker=dict(size=4, color='red', opacity=0.8, symbol='x')), row=1, col=1)
        fig.add_trace(go.Scatter3d(x=cache['fn_coords'][:, 2] if len(cache['fn_coords']) > 0 else [], y=cache['fn_coords'][:, 1] if len(cache['fn_coords']) > 0 else [], z=cache['fn_coords'][:, 0] if len(cache['fn_coords']) > 0 else [], mode='markers', name='FN (Missing)', marker=dict(size=4, color='cyan', opacity=0.8, symbol='diamond')), row=1, col=1)
        
        # Subplot 2 (1,2): XY-MIP
        fig.add_trace(go.Surface(x=x_grid_z, y=y_grid_z, z=np.zeros_like(mip_z), surfacecolor=mip_z, colorscale='Gray', showscale=False, hoverinfo='x+y+z', name='MIP Z'), row=1, col=2)
        fig.add_trace(go.Scatter3d(x=cache['tp_coords'][:, 2] if len(cache['tp_coords']) > 0 else [], y=cache['tp_coords'][:, 1] if len(cache['tp_coords']) > 0 else [], z=np.full(len(cache['tp_coords']), 0.1) if len(cache['tp_coords']) > 0 else [], mode='markers', name='TP XY', marker=dict(size=4, color='lime', opacity=0.9), showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter3d(x=cache['fp_coords'][:, 2] if len(cache['fp_coords']) > 0 else [], y=cache['fp_coords'][:, 1] if len(cache['fp_coords']) > 0 else [], z=np.full(len(cache['fp_coords']), 0.1) if len(cache['fp_coords']) > 0 else [], mode='markers', name='FP XY', marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter3d(x=cache['fn_coords'][:, 2] if len(cache['fn_coords']) > 0 else [], y=cache['fn_coords'][:, 1] if len(cache['fn_coords']) > 0 else [], z=np.full(len(cache['fn_coords']), 0.1) if len(cache['fn_coords']) > 0 else [], mode='markers', name='FN XY', marker=dict(size=4, color='cyan', opacity=0.9), showlegend=False), row=1, col=2)

        # Subplot 3 (2,1): XZ-MIP
        fig.add_trace(go.Surface(x=x_grid_y, y=np.zeros_like(x_grid_y), z=z_grid_y, surfacecolor=mip_y, colorscale='Gray', showscale=False, hoverinfo='x+y+z', name='MIP Y'), row=2, col=1)
        fig.add_trace(go.Scatter3d(x=cache['tp_coords'][:, 2] if len(cache['tp_coords']) > 0 else [], y=np.full(len(cache['tp_coords']), 0.1) if len(cache['tp_coords']) > 0 else [], z=cache['tp_coords'][:, 0] if len(cache['tp_coords']) > 0 else [], mode='markers', name='TP XZ', marker=dict(size=4, color='lime', opacity=0.9), showlegend=False), row=2, col=1)
        fig.add_trace(go.Scatter3d(x=cache['fp_coords'][:, 2] if len(cache['fp_coords']) > 0 else [], y=np.full(len(cache['fp_coords']), 0.1) if len(cache['fp_coords']) > 0 else [], z=cache['fp_coords'][:, 0] if len(cache['fp_coords']) > 0 else [], mode='markers', name='FP XZ', marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False), row=2, col=1)
        fig.add_trace(go.Scatter3d(x=cache['fn_coords'][:, 2] if len(cache['fn_coords']) > 0 else [], y=np.full(len(cache['fn_coords']), 0.1) if len(cache['fn_coords']) > 0 else [], z=cache['fn_coords'][:, 0] if len(cache['fn_coords']) > 0 else [], mode='markers', name='FN XZ', marker=dict(size=4, color='cyan', opacity=0.9), showlegend=False), row=2, col=1)

        # Subplot 4 (2,2): YZ-MIP
        fig.add_trace(go.Surface(x=np.zeros_like(y_grid_x), y=y_grid_x, z=z_grid_x, surfacecolor=mip_x, colorscale='Gray', showscale=False, hoverinfo='x+y+z', name='MIP X'), row=2, col=2)
        fig.add_trace(go.Scatter3d(x=np.full(len(cache['tp_coords']), 0.1) if len(cache['tp_coords']) > 0 else [], y=cache['tp_coords'][:, 1] if len(cache['tp_coords']) > 0 else [], z=cache['tp_coords'][:, 0] if len(cache['tp_coords']) > 0 else [], mode='markers', name='TP YZ', marker=dict(size=4, color='lime', opacity=0.9), showlegend=False), row=2, col=2)
        fig.add_trace(go.Scatter3d(x=np.full(len(cache['fp_coords']), 0.1) if len(cache['fp_coords']) > 0 else [], y=cache['fp_coords'][:, 1] if len(cache['fp_coords']) > 0 else [], z=cache['fp_coords'][:, 0] if len(cache['fp_coords']) > 0 else [], mode='markers', name='FP YZ', marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False), row=2, col=2)
        fig.add_trace(go.Scatter3d(x=np.full(len(cache['fn_coords']), 0.1) if len(cache['fn_coords']) > 0 else [], y=cache['fn_coords'][:, 1] if len(cache['fn_coords']) > 0 else [], z=cache['fn_coords'][:, 0] if len(cache['fn_coords']) > 0 else [], mode='markers', name='FN YZ', marker=dict(size=4, color='cyan', opacity=0.9), showlegend=False), row=2, col=2)

        # 3Dカメラの設定
        scene_config = dict(
            xaxis=dict(title='X (Width, um)', range=[0, nx * scale[2]]),
            yaxis=dict(title='Y (Height, um)', range=[0, ny * scale[1]]),
            zaxis=dict(title='Z (Depth, um)', range=[0, nz * scale[0]]),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        )
        
        fig.update_layout(
            title_text=f"Cell Detection Interactive 3D MIP Overlays Dashboard (Frame {t})",
            scene1=scene_config,
            scene2=scene_config,
            scene3=scene_config,
            scene4=scene_config,
            autosize=True,
            height=850,
            margin=dict(l=20, r=20, b=20, t=80),
            showlegend=True
        )
        return fig

    # --- widgets 構成 ---
    # プロットを出力するための汎用 Output ウィジェット
    plot_output = widgets.Output()
    eval_html = widgets.HTML(value=frame_data_cache[0]['eval_text'])
    
    frame_slider = widgets.IntSlider(
        value=0, min=0, max=n_frames-1, step=1,
        description='Frame Select:',
        layout=widgets.Layout(width='60%')
    )
    
    frame_input = widgets.IntText(
        value=0,
        description='',
        layout=widgets.Layout(width='80px')
    )
    
    # スライダーとインプットボックスの JavaScript 双方向リンク
    widgets.jslink((frame_slider, 'value'), (frame_input, 'value'))

    # スライダー更新時の Python イベントハンドラ
    def on_frame_change(change):
        t = change['new']
        cache = frame_data_cache[t]
        
        # 評価HTMLテキストを更新
        eval_html.value = cache['eval_text']
        
        # 新しい Figure の生成と再描画
        new_fig = create_figure_for_frame(t)
        
        with plot_output:
            plot_output.clear_output(wait=True)
            new_fig.show()

    # スライダーの変更監視を設定
    frame_slider.observe(on_frame_change, names='value')

    # 初期描画を実行 (Frame 0)
    initial_fig = create_figure_for_frame(0)
    with plot_output:
        initial_fig.show()

    # 操作パネルの組み立て
    controls = widgets.HBox([
        frame_slider, 
        widgets.Label(value='Input Frame:'), 
        frame_input
    ], layout=widgets.Layout(align_items='center', margin='10px 0px'))
    
    # [plot_output, controls, eval_html] の順に垂直配置
    dashboard = widgets.VBox([
        plot_output,
        controls,
        eval_html
    ])
    
    # ダッシュボードを表示
    display(dashboard)


## 【ステップ 3 & 4】トラッキング評価とトラッカーの向上

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

from scipy.spatial.distance import cdist

# btrack のインポートとフォールバック判定をノートブック内で直接実行
try:
    import btrack
    import btrack.datasets
    from btrack.btypes import PyTrackObject
    HAS_BTRACK = True
except ImportError as e:
    print(f"Warning: Failed to import btrack ({e}). Falling back to Nearest Neighbor tracking.")
    HAS_BTRACK = False

def run_nearest_neighbor_tracking(nodes_df, max_search_radius=25.0):
    """
    btrackが利用できない場合のフォールバック最近傍マッチングトラッキング。
    """
    if len(nodes_df) == 0:
        return []
    
    # 時間順にソート
    nodes_df = nodes_df.sort_values('t')
    frames = sorted(nodes_df['t'].unique())
    
    edges = []
    # 連続するフレーム間で貪欲マッチング
    for idx, t in enumerate(frames[:-1]):
        t_next = frames[idx+1]
        if t_next != t + 1:
            continue
            
        df_prev = nodes_df[nodes_df['t'] == t]
        df_curr = nodes_df[nodes_df['t'] == t_next]
        
        if len(df_prev) == 0 or len(df_curr) == 0:
            continue
            
        coords_prev = df_prev[['z', 'y', 'x']].values
        coords_curr = df_curr[['z', 'y', 'x']].values
        
        dists = cdist(coords_prev, coords_curr)
        
        used_curr = set()
        prev_indices = df_prev['node_id'].values
        curr_indices = df_curr['node_id'].values
        
        for i in range(len(coords_prev)):
            js = np.argsort(dists[i])
            for j in js:
                if j not in used_curr and dists[i, j] <= max_search_radius:
                    edges.append({
                        'source_id': int(prev_indices[i]),
                        'target_id': int(curr_indices[j])
                    })
                    used_curr.add(j)
                    break
    return edges

def run_btrack_tracking(nodes_df, max_search_radius=25.0):
    """
    btrack ベイジアン追跡 + ILP 最適化の実行。
    btrackがインストールされていないか、DLL読み込みエラー等で失敗した場合は、自動的に最近傍フォールバックを実行。
    """
    if len(nodes_df) == 0:
        return []
    
    if not HAS_BTRACK:
        return run_nearest_neighbor_tracking(nodes_df, max_search_radius)
        
    try: 
        # 1. オブジェクト変換と btrack 用シーケンシャル ID の割当て
        objects = []
        id_map = {}
        for idx, row in enumerate(nodes_df.itertuples()):
            obj = PyTrackObject()
            obj.ID = idx
            obj.t = int(row.t)
            obj.z = float(row.z)
            obj.y = float(row.y)
            obj.x = float(row.x)
            objects.append(obj)
            id_map[idx] = int(row.node_id)
 
        # 2. Configure Bayesian Tracker
        with btrack.BayesianTracker() as tracker:
            cell_cfg = btrack.datasets.cell_config()
            tracker.configure(cell_cfg)
            tracker.max_search_radius = max_search_radius
            
            tracker.append(objects)
            tracker.track()
            tracker.optimize()
 
            tracks = tracker.tracks
 
            # 3. エッジの抽出と元の node_id への逆引きマッピング
            edges = []
            for track in tracks:
                # 同一トラック内のエッジ
                for i in range(len(track.dummy) - 1):
                    if not track.dummy[i] and not track.dummy[i+1]:
                        src_id = id_map[track.refs[i]]
                        tgt_id = id_map[track.refs[i+1]]
                        edges.append({'source_id': src_id, 'target_id': tgt_id})
                
                # 細胞分裂によるエッジ
                if track.parent > 0:
                    parent_track = [t for t in tracks if t.ID == track.parent]
                    if parent_track and not parent_track[0].dummy[-1] and not track.dummy[0]:
                        src_id = id_map[parent_track[0].refs[-1]]
                        tgt_id = id_map[track.refs[0]]
                        edges.append({'source_id': src_id, 'target_id': tgt_id})
        return edges
    except Exception as e:
        print(f"Error during btrack tracking ({e}). Falling back to Nearest Neighbor tracking.")
        return run_nearest_neighbor_tracking(nodes_df, max_search_radius)

def evaluate_tracking_only(pred_edges, gt_graph, gt_nodes_df, scale):
    """
    ステップ3: トラッキング単体の評価 (GTノードを直接入力)
    """
    from tracking_cellmot.metrics import evaluate
    from tracksdata.graph import IndexedRXGraph
    
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in gt_nodes_df.itertuples():
        pred_graph.add_node(
            attrs={
                't': int(row.t),
                'z': float(row.z),
                'y': float(row.y),
                'x': float(row.x)
            },
            index=int(row.node_id)
        )
        
    for edge in pred_edges:
        pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']), attrs={})
        
    res = evaluate(pred_graph, gt_graph, scale=scale)
    
    edge_denom = res.edge_tp + res.edge_fp + res.edge_fn
    edge_jaccard = res.edge_tp / edge_denom if edge_denom > 0 else 1.0
    
    return {
        'edge_jaccard': edge_jaccard,
        'edge_tp': res.edge_tp,
        'edge_fp': res.edge_fp,
        'edge_fn': res.edge_fn
    }

print("Tracking evaluation functions defined.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell execution completed. (Elapsed: {_cell_elapsed:.2f}s)")


## 【ステップ 5 & 6】統合評価と間引き (Pruning) の実行

検出予測ノードを用いたトラッキングの実行と公式評価（統合評価）、および不要なノードやエッジを削る間引き (Pruning) 処理を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

def evaluate_complete(pred_nodes_df, pruned_edges, gt_graph, scale):
    """
    ステップ5: 統合評価 (検出予測ノードを用いたトラッキング実行と公式評価)
    """
    from tracking_cellmot.metrics import evaluate
    from tracksdata.graph import IndexedRXGraph
    
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node(
            attrs={
                't': int(row.t),
                'z': float(row.z),
                'y': float(row.y),
                'x': float(row.x)
            },
            index=int(row.node_id)
        )
        
    for edge in pruned_edges:
        pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']), attrs={})
        
    res = evaluate(pred_graph, gt_graph, scale=scale)
    return res, pred_graph

def prune_tracks(nodes_df, edges):
    """
    ステップ6: 間引き (Pruning) / 後処理の実行（プレースホルダー）
    """
    print("Pruning placeholder: returning input nodes and edges without modifications.")
    return nodes_df, edges

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Complete evaluation and pruning functions defined. (Elapsed: {_cell_elapsed:.2f}s)")


## ベースライン全体の実行とテスト

データセットを読み込み、これまでに実装した検出器の検証を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# データ読み込みとmain()テスト実行
import sys
import glob
import zarr
import numpy as np
import pandas as pd
from tracking_cellmot.io import open_dataset
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'input', 'train'))
dataset_paths = glob.glob(os.path.join(DATA_DIR, '*.zarr')) + glob.glob(os.path.join(DATA_DIR, '*.geff'))
if not dataset_paths:
    dataset_paths = glob.glob('/kaggle/input/**/train/*.zarr', recursive=True) + glob.glob('/kaggle/input/**/train/*.geff', recursive=True)

if not dataset_paths:
    raise FileNotFoundError("Error: No .zarr or .geff datasets found in the data directories.")

target_dataset_path = dataset_paths[0]
print(f'Target dataset: {target_dataset_path}')

# GTのロード (失敗時はエラー終了)
try:
    ds_gt = open_dataset(target_dataset_path, normalize=True, require_tracks=True, device='cpu')
except Exception as e:
    print(f"Error: Failed to load dataset {target_dataset_path}. Reason: {e}", file=sys.stderr)
    raise e

gt_graph = ds_gt.tracks
scale = ds_gt.scale

# 1. 検出評価
print('\n--- Running Node Detection Evaluation ---')
pred_nodes = detect_nodes_baseline(target_dataset_path, max_frames=3)  # トラッキング評価用に3フレーム実行
det_res = evaluate_detection_only(pred_nodes, gt_graph, scale=scale)

print("\n=================== DETAILED DETECTION EVALUATION ===================")
print(f"  TP (Matched predicted nodes): {det_res['tp']}")
print(f"  FP (Extra predicted nodes):   {det_res['fp']}")
print(f"  FN (Missed GT nodes in range):{det_res['fn']} (GT in range total: {det_res['gt_total_in_range']})")
print(f"  Detection Precision:          {det_res['precision']:.4f}  [ Formula: TP / T_pred = {det_res['tp']} / {det_res['pred_total']} ]")
print(f"  Detection Recall (Local):     {det_res['recall_local']:.4f}  [ Formula: TP / GT_in_range = {det_res['tp']} / {det_res['gt_total_in_range']} ]")
print(f"  Detection Recall (Global):    {det_res['recall_global']:.4f}  [ Formula: TP / GT_total = {det_res['tp']} / {gt_graph.num_nodes()} ]")
print(f"  Detection F1-Score (Local):   {det_res['f1_local']:.4f}")
print(f"  Detection F1-Score (Global):  {det_res['f1_global']:.4f}")
print("=====================================================================\n")

# 可視化の実行 (MIP画像の取得とダッシュボード表示)
# 可視化の実行 (MIP画像の取得とダッシュボード表示)
print('\n--- Visualizing cell centroids with interactive dashboard ---')
img_4d = ds_gt.image
if hasattr(img_4d, 'numpy'):
    img_4d = img_4d.numpy()
    
n_frames_eval = int(pred_nodes['t'].max() + 1)
plot_detection_dashboard(img_4d[:n_frames_eval], pred_nodes, gt_graph, scale=scale)

# 2. トラッキング評価 (GTノードを入力)
print('\n--- Running Tracking Evaluation with GT Nodes ---')
gt_nodes_pl = gt_graph.node_attrs(attr_keys=['node_id', 't', 'z', 'y', 'x'])
gt_nodes_df = gt_nodes_pl.to_pandas()

# btrack（またはNN）実行 (ノートブック内で定義した run_btrack_tracking を直接コール)
edges = run_btrack_tracking(gt_nodes_df, max_search_radius=25.0)
track_res = evaluate_tracking_only(edges, gt_graph, gt_nodes_df, scale=scale)
print(f'Edge Jaccard (GT Nodes): {track_res["edge_jaccard"]:.4f} (TP={track_res["edge_tp"]}, FP={track_res["edge_fp"]}, FN={track_res["edge_fn"]})')

# 3. 統合評価 (検出予測ノードを用いたトラッキング実行)
print('\n--- Running Complete End-to-End Evaluation ---')
complete_edges = run_btrack_tracking(pred_nodes, max_search_radius=25.0)

# 4. 間引き (Pruning) / 後処理の実行
pruned_nodes, pruned_edges = prune_tracks(pred_nodes, complete_edges)

# 5. 統合スコア評価とペナルティ（Adjusted Jaccard）の算出
complete_res, pred_graph = evaluate_complete(pruned_nodes, pruned_edges, gt_graph, scale=scale)

# GEFF のメタデータから estimated_number_of_nodes の取得
from pathlib import Path
from geff import GeffMetadata
target_path_obj = Path(target_dataset_path)
if target_path_obj.suffix in (".zarr", ".geff"):
    geff_path = target_path_obj.parent / f"{target_path_obj.stem}.geff"
else:
    geff_path = target_path_obj.parent / f"{target_path_obj.name}.geff"

n_total_estimated = float("nan")
try:
    meta = GeffMetadata.read(geff_path)
    val = (meta.extra or {}).get("estimated_number_of_nodes")
    if val is not None:
        n_total_estimated = float(val)
except Exception as e:
    print(f"Warning: Could not read estimated_number_of_nodes from GEFF metadata: {e}")

gt_nodes_total = gt_graph.num_nodes()

print("\n=================== NODE COUNT METRICS ===================")
print(f"  Metadata 'estimated_number_of_nodes' (from GEFF): {n_total_estimated}")
print(f"  GT Graph Total Nodes (gt_graph.num_nodes()): {gt_nodes_total}")
print("==========================================================\n")

if np.isnan(n_total_estimated):
    raise ValueError(
        "CRITICAL ERROR: 'estimated_number_of_nodes' was not found in GEFF metadata attributes! "
        f"Evaluation stopped to prevent incorrect penalty metrics. (For reference, gt_graph.num_nodes() = {gt_nodes_total})"
    )

from tracking_cellmot.metrics import per_sample_metrics, node_recall
rec_val = node_recall(pred_graph, gt_graph)
p_metrics = per_sample_metrics(complete_res, n_total=n_total_estimated, node_recall=rec_val)

print("\n=================== INTEGRATED EVALUATION RESULTS ===================")
print(f"  Estimated True Nodes (n_total, Dataset Total GT): {n_total_estimated}")
print(f"  Predicted Nodes (T_pred, 3-Frame Total):          {p_metrics['num_pred_nodes']}")
print(f"  Node Recall (matched ratio):                      {p_metrics['node_recall']:.4f}  [ Matched ({det_res['tp']}) / Dataset Total GT ({int(n_total_estimated)}) ]")

print(f"  Edge Jaccard:                                     {p_metrics['edge_jaccard']:.4f}  (TP={p_metrics['edge_tp']}, FP={p_metrics['edge_fp']}, FN={p_metrics['edge_fn']})")
print(f"    [ Formula: TP / (TP + FP + FN) = {p_metrics['edge_tp']} / ({p_metrics['edge_tp']} + {p_metrics['edge_fp']} + {p_metrics['edge_fn']}) ]")

print(f"  Node Ratio (extra nodes ratio):                  {p_metrics['total_node_ratio']:.4f}")
print(f"    [ Formula: (T_pred - n_total) / n_total = ({p_metrics['num_pred_nodes']} - {int(n_total_estimated)}) / {int(n_total_estimated)} ]")

ratio_val = p_metrics['total_node_ratio']
penalty_val = 1.0 / max(1.0, ratio_val - 3.0)
print(f"  Penalty Coefficient (P):                          {penalty_val:.4f}")
print(f"    [ Formula: 1.0 / max(1.0, Ratio - 3.0) = 1.0 / ({ratio_val:.4f} - 3.0) ]")

print(f"  Adjusted Edge Jaccard:                            {p_metrics['adj_edge_jaccard']:.4f}")
print(f"    [ Formula: Edge Jaccard ({p_metrics['edge_jaccard']:.4f}) * Penalty ({penalty_val:.4f}) -> Adjusted ]")

div_denom = p_metrics['division_tp'] + p_metrics['division_fp'] + p_metrics['division_fn']
div_jaccard = p_metrics['division_tp'] / div_denom if div_denom > 0 else 0.0
print(f"  Division Jaccard:                                 {div_jaccard:.4f}  (TP={p_metrics['division_tp']}, FP={p_metrics['division_fp']}, FN={p_metrics['division_fn']})")
print(f"    [ Formula: TP / (TP + FP + FN) ]")

print(f"  COMBINED SCORE:                                   {p_metrics['adj_edge_jaccard'] + 0.1 * div_jaccard:.4f}")
print(f"    [ Formula: Adjusted Edge Jaccard ({p_metrics['adj_edge_jaccard']:.4f}) + 0.1 * Division Jaccard ({div_jaccard:.4f}) ]")
print("=====================================================================\n")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Pipeline detection evaluation completed. (Elapsed: {_cell_elapsed:.2f}s)")
